<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Los Exámenes de Práctica Justos: Pacientes, Calendario y el Concurso de Talentos 🎤
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/Para%20Dummies/01_Validacion_Cruzada_Avanzada_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 (Validación cruzada avanzada)](../01_Validacion_Cruzada_Avanzada.ipynb). La validación cruzada es la idea de **hacer varios exámenes de práctica** en lugar de uno solo, para saber cuánto sabe un modelo. Pero hay **tres trampas** que hacen que los exámenes de práctica **mientan**: preguntas repetidas (pacientes), mirar el futuro (calendario) y elegir al ganador con el mismo juez (concurso de talentos). Aquí **no hay fórmulas**: solo *NumPy* y gráficas.

Al terminar podrás explicar:
1. Por qué **un solo simulacro** no basta y qué ganan los simulacros **repetidos**.
2. Por qué hay que **separar por paciente** (y no por foto) cuando hay varias filas de la misma persona.
3. Por qué con datos de **calendario** no se puede «mezclar y partir».
4. Por qué elegir al **mejor concursante** con un juez y evaluarlo con **el mismo juez** da una nota inflada, y qué es la validación **anidada**.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../01_Validacion_Cruzada_Avanzada.ipynb). Antes: [El examen sorpresa, la moneda y el pronosticador (Dummies)](00_Evaluacion_Seleccion_Dummies.ipynb).

---
## 1. Un solo simulacro no basta 📋

Un estudiante sabe **80 %** de la materia. Le hacemos un simulacro de **40 preguntas**, pero las preguntas que le tocan son una muestra al azar: a veces le salen las que sabe, a veces las que no. Su nota en **un** simulacro **cambia** según la suerte. Si hacemos **5 simulacros distintos** y promediamos, la nota se estabiliza. Eso es la **validación cruzada repetida**: varios exámenes de práctica, promedio de todos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
SABE = 0.80                                                    # el estudiante sabe el 80 % de la materia (el valor "verdadero")
def simulacro(n_preguntas=40):
    return np.mean(rng.random(n_preguntas) < SABE)             # nota de UN simulacro de n preguntas

un_simulacro = np.array([simulacro() for _ in range(2000)])
promedio_de_5 = np.array([np.mean([simulacro() for _ in range(5)]) for _ in range(2000)])
print(f"UN simulacro:        la nota varía entre {np.percentile(un_simulacro, 2.5):.0%} y {np.percentile(un_simulacro, 97.5):.0%} (el 95 % de las veces)")
print(f"Promedio de 5 simulacros: la nota varía entre {np.percentile(promedio_de_5, 2.5):.0%} y {np.percentile(promedio_de_5, 97.5):.0%}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(un_simulacro, bins=np.linspace(0.5, 1, 30), alpha=0.75, color="#dc2626", density=True, label="Un solo simulacro")
ax.hist(promedio_de_5, bins=np.linspace(0.5, 1, 30), alpha=0.75, color="#16a34a", density=True, label="Promedio de 5 simulacros")
ax.axvline(SABE, color="black", ls="--", label="Lo que de verdad sabe (80 %)")
ax.set_xlabel("Nota estimada"); ax.set_ylabel("Densidad"); ax.legend(fontsize=8)
ax.set_title("Repetir los simulacros estabiliza la estimación", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert promedio_de_5.std() < un_simulacro.std() * 0.55

---
**Lección:** un simulacro suelto puede dar 70 % o 90 % solo por **qué preguntas tocaron**. El promedio de varios es mucho más estable (aproximadamente la mitad de «temblor» con 5 simulacros). Pero ojo: repetir **no** vuelve el estudiante más sabio ni convierte la estimación en «la verdad»: solo quita el ruido de **qué preguntas tocaron**.

## 2. Cuidado con las fotos del mismo paciente 🏥

Tenemos **30 pacientes**, con **10 fotos** cada uno. Cada paciente tiene su «cara» (rasgos propios) y está **enfermo o sano**. Un programa aprende a decir «enfermo/sano» **mirando las fotos**. Hay dos formas de hacer el examen de práctica:

- **Mezclar todas las fotos y partir**: algunas fotos de Ana quedan en el estudio y otras en el examen. El programa solo tiene que **reconocer la cara de Ana** y copiar su diagnóstico.
- **Separar por paciente**: **todas** las fotos de Ana están o en el estudio o en el examen. El programa debe diagnosticar a pacientes **que nunca vio**, que es lo que pasará en la vida real.

In [ ]:
rng = np.random.default_rng(5)
pacientes, fotos = 40, 10
enfermo = rng.integers(0, 2, pacientes)
cara = rng.normal(size=(pacientes, 8))                                    # los rasgos propios de cada paciente
X = np.vstack([cara[p] + rng.normal(0, 0.4, (fotos, 8)) for p in range(pacientes)])        # cada foto = cara + pequeñas variaciones
y = np.repeat(enfermo, fotos); quien = np.repeat(np.arange(pacientes), fotos)             # etiqueta y 'de quién es' cada foto

def vecino_mas_parecido(X_estudio, y_estudio, X_examen):
    d = ((X_examen[:, None, :] - X_estudio[None, :, :]) ** 2).sum(axis=2)
    return y_estudio[d.argmin(axis=1)]

resultados = {"mezclando todas las fotos": [], "separando por paciente": []}
for repeticion in range(30):
    r = np.random.default_rng(repeticion)
    # (a) Mezclar fotos: 30 % de las fotos al examen, al azar
    idx = r.permutation(len(y)); examen, estudio = idx[: len(y) * 3 // 10], idx[len(y) * 3 // 10:]
    resultados["mezclando todas las fotos"].append(np.mean(vecino_mas_parecido(X[estudio], y[estudio], X[examen]) == y[examen]))
    # (b) Separar por paciente: 30 % de los PACIENTES (con todas sus fotos) al examen
    pac = r.permutation(pacientes); pac_examen = pac[: pacientes * 3 // 10]
    es_examen = np.isin(quien, pac_examen)
    resultados["separando por paciente"].append(np.mean(vecino_mas_parecido(X[~es_examen], y[~es_examen], X[es_examen]) == y[es_examen]))

medias = {k: np.mean(v) for k, v in resultados.items()}
for k, v in medias.items(): print(f"Examen de práctica {k}: nota media = {v:.0%}")
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(list(medias), list(medias.values()), color=["#dc2626", "#16a34a"], yerr=[np.std(v) for v in resultados.values()], capsize=6)
ax.axhline(0.5, color="black", ls="--", label="Adivinar con una moneda"); ax.set_ylim(0, 1.1); ax.set_ylabel("Fracción de diagnósticos correctos"); ax.legend()
ax.set_title("Mismo programa, mismos datos: la nota depende de CÓMO se parte", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert medias["mezclando todas las fotos"] > 0.9 and medias["separando por paciente"] < 0.7

---
**Lección:** mezclando fotos, el programa saca **más del 90 %** (¡solo reconoce caras!). Separando por paciente, saca **alrededor de 50 %** porque en estos datos **no hay nada real que aprender** más allá de la cara. La nota que importa es la segunda. **Regla:** si hay **varias filas de la misma persona, cliente, sensor o documento**, se parte **por grupo**, no por fila.

## 3. No se puede adivinar el pasado con el futuro ⏳

Tienes las ventas de **200 días** (con tendencia al alza y variaciones suaves). Quieres saber qué tan bien **predices** las ventas. Dos exámenes de práctica:

- **Tapar días al azar** y adivinarlos con los **vecinos** (el día de antes **y el de después**). Es fácil: los días vecinos se parecen.
- **Predecir el futuro**: ver solo hasta el día *t* y adivinar **una semana después** (con lo que hay **hasta ese momento**). Es lo que harás en la vida real.

In [ ]:
rng = np.random.default_rng(6)
dias = 200
ruido = np.zeros(dias)
for t in range(1, dias): ruido[t] = 0.85 * ruido[t - 1] + rng.normal(0, 0.5)         # variaciones suaves: el de hoy se parece al de ayer
ventas = 0.05 * np.arange(dias) + np.sin(np.arange(dias) / 6) * 2 + ruido            # tendencia + estaciones + variaciones

# (a) Tapar días al azar y rellenar con el promedio del día anterior y el posterior (¡mira el futuro!)
tapados = rng.choice(np.arange(1, dias - 1), 60, replace=False)
pred_a = (ventas[tapados - 1] + ventas[tapados + 1]) / 2
error_a = np.mean(np.abs(ventas[tapados] - pred_a))
# (b) Pronosticar la SEMANA SIGUIENTE (7 días hacia adelante) usando SOLO el pasado: repetir lo que se vendió hace una semana
dias_pron = np.arange(100, dias)
pred_b = ventas[dias_pron - 7]
error_b = np.mean(np.abs(ventas[dias_pron] - pred_b))
print(f"Error al 'tapar días y mirar vecinos de AMBOS lados': {error_a:.2f}")
print(f"Error al 'pronosticar 7 días hacia adelante con solo el pasado': {error_b:.2f}  ({error_b / error_a:.1f} veces mayor)")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ventas, color="#475569", lw=1.5, label="Ventas reales")
ax.scatter(tapados, ventas[tapados], color="#dc2626", s=18, zorder=5, label="Días tapados (examen que 'hace trampa')")
ax.axvspan(100, dias, color="#16a34a", alpha=0.1, label="Zona de pronóstico real (solo pasado)")
ax.set_xlabel("Día"); ax.set_ylabel("Ventas"); ax.legend(fontsize=8, loc="upper left")
ax.set_title("Mezclar los días hace el examen más fácil de lo que será la realidad", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert error_b > 1.5 * error_a

---
**Lección:** con los vecinos de **ambos lados** el error es pequeño; prediciendo **hacia adelante** es claramente mayor (y crece cuanto más lejos miras). El primer examen **mide qué tan bien rellenas huecos**, no qué tan bien **pronosticas**. Con datos de calendario hay que practicar **siempre hacia adelante**: estudiar con el pasado y examinar con lo que viene después (**validación *walk-forward***, «caminar hacia adelante»).

## 4. El concurso de talentos: el juez no puede ser el que elige 🎤

**50 concursantes** con **exactamente el mismo talento** (todos «medio buenos»). Un juez les pone nota, pero cada nota tiene algo de **suerte** (un día bueno o malo). Elegimos al **ganador**: el que sacó la nota más alta. ¿Cuánto talento tiene realmente el ganador? **El mismo que los demás**: su nota alta fue mucha suerte. Si **otro juez** lo evalúa de nuevo, baja.

Esto pasa cuando un modelo «gana» entre muchas combinaciones de ajustes (hiperparámetros): su **mejor nota** está **inflada**. La solución se llama **validación cruzada anidada**: **un juez elige** al ganador y **otro juez distinto** lo califica.

In [ ]:
rng = np.random.default_rng(7)
concursantes, repeticiones = 50, 3000
talento = 0.60                                                   # TODOS tienen el mismo talento real: 60 %
juez_1 = talento + rng.normal(0, 0.05, (repeticiones, concursantes))      # nota del juez que ELIGE (con suerte)
juez_2 = talento + rng.normal(0, 0.05, (repeticiones, concursantes))      # nota de OTRO juez independiente

ganador = juez_1.argmax(axis=1)                                  # el que más nota sacó con el juez 1
nota_inflada = juez_1.max(axis=1).mean()                         # lo que reportaría quien usa SOLO el juez que eligió
nota_justa = juez_2[np.arange(repeticiones), ganador].mean()     # la nota del ganador con el otro juez
print(f"Talento real de TODOS los concursantes: {talento:.0%}")
print(f"Nota del ganador según el juez que lo eligió: {nota_inflada:.1%}   <- inflada por la suerte")
print(f"Nota del MISMO ganador con otro juez:        {nota_justa:.1%}   <- la verdad (≈ su talento real)")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(["Talento\nreal", "Nota con el juez\nque lo eligió", "Nota con\notro juez"], [talento, nota_inflada, nota_justa], color=["#475569", "#dc2626", "#16a34a"])
ax.set_ylim(0.5, 0.75); ax.set_ylabel("Nota"); ax.set_title("El ganador no es tan bueno como parece (maldición del ganador)", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert nota_inflada > talento + 0.05 and abs(nota_justa - talento) < 0.01

---
##### 🛠️ Práctica 1: ¿Y si en el concurso hay muchos menos (o muchos más) concursantes?

En el concurso de talentos los 50 concursantes eran **igual de buenos** y el ganador salió inflado por pura suerte. ¿Depende esa inflación de **cuántos** compiten? Usa las notas ya simuladas: `juez_1` (el juez que elige), `juez_2` (otro juez independiente), `talento`, `repeticiones` y `np`. Para grupos de 1, 2, 5, 10 y 50 concursantes (las primeras columnas de `juez_1`):

1. Calcula la nota inflada: `juez_1[:, :n].max(axis=1).mean()`.
2. Mira **quién ganó** (`juez_1[:, :n].argmax(axis=1)`) y calcula su nota con el otro juez: `juez_2[np.arange(repeticiones), ganador_n].mean()`.
3. Guarda en dos diccionarios, `inflada` y `con_otro_juez`, esas notas (clave = número de concursantes) y calcula `exceso[n] = inflada[n] - talento`. Imprime una línea por tamaño.
4. Comprueba con `assert` que el exceso **crece** al haber más concursantes, que con 1 solo concursante no hay inflación (exceso ≈ 0) y que la nota con el otro juez siempre queda cerca del talento real.

In [ ]:
# Escribe tu código aquí

# tamanos = [1, 2, 5, 10, 50]
# inflada, con_otro_juez, exceso = {}, {}, {}
# for n in tamanos:
#     inflada[n] = juez_1[:, :n].max(axis=1).mean()
#     ganador_n = juez_1[:, :n].argmax(axis=1)
#     con_otro_juez[n] = juez_2[np.arange(repeticiones), ganador_n].mean()
#     exceso[n] = ...
#     print(f"{n:>2} concursantes: inflada = {inflada[n]:.3f} | con otro juez = {con_otro_juez[n]:.3f} | exceso = {exceso[n]:+.3f}")


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
tamanos = [1, 2, 5, 10, 50]
inflada, con_otro_juez, exceso = {}, {}, {}
for n in tamanos:
    inflada[n] = juez_1[:, :n].max(axis=1).mean()                           # nota del ganador segun el juez que lo eligio
    ganador_n = juez_1[:, :n].argmax(axis=1)                                # quien gano en cada repeticion
    con_otro_juez[n] = juez_2[np.arange(repeticiones), ganador_n].mean()    # la nota del MISMO ganador con otro juez
    exceso[n] = inflada[n] - talento                                        # cuanto se infla por pura suerte
    print(f"{n:>2} concursantes: inflada = {inflada[n]:.3f} | con otro juez = {con_otro_juez[n]:.3f} | exceso = {exceso[n]:+.3f}")

excesos = [exceso[n] for n in tamanos]
assert all(a < b for a, b in zip(excesos[:-1], excesos[1:]))                # mas concursantes -> mas suerte acumulada en el maximo
assert abs(exceso[1]) < 0.01                                                # con uno solo no hay 'ganador' que se infle
assert all(abs(con_otro_juez[n] - talento) < 0.01 for n in tamanos)         # el otro juez no se deja enganar
```
</details>

---


---
## Resumen en una frase 🎯

> **Los exámenes de práctica deben parecerse al uso real: varios simulacros mejor que uno, separar por paciente (grupo) y no por foto, mirar siempre hacia adelante con datos de calendario, y usar un juez distinto para elegir al ganador y para calificarlo (validación anidada).**

### Lo que aprendiste hoy:
- ✅ **Un simulacro** es ruidoso; **varios** lo estabilizan, pero no te dicen toda la verdad.
- ✅ **Grupos**: con varias fotos por paciente, mezclar da **> 90 %** (solo reconoce caras); separar por paciente da **≈ 50 %** (el valor real en estos datos).
- ✅ **Calendario**: tapar días al azar es un examen **demasiado fácil**; pronosticar hacia adelante tiene un error claramente mayor.
- ✅ **Concurso de talentos**: el ganador de 50 iguales sacó ≈ 70 % con su juez y ≈ 60 % con otro: **la mejor nota está inflada**; la validación **anidada** usa un juez distinto.

> 🎓 **Siguiente paso:** explora el [cuaderno estándar](../01_Validacion_Cruzada_Avanzada.ipynb) y continúa con [Buscar las llaves: rejilla, azar y zoom (Dummies)](02_Random_Grid_Search_Dummies.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué es «trampa» hacer el examen de práctica con **fotos del mismo paciente** mezcladas entre estudio y examen? ¿Qué debería hacerse?
2. Quieres predecir las ventas de **la próxima semana**. ¿Por qué no sirve mezclar los días para validar? ¿Cómo practicarías «hacia adelante»?
3. Un compañero probó 100 recetas de ajustes, se quedó con la que dio la mejor nota y dice «mi modelo saca 92 %». ¿Qué le responderías y qué le propondrías para obtener una nota más honesta?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>